In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import os

BASE = "https://stats.ioinformatics.org/"
RESULT_CLASSES = {"gold", "silver", "bronze", "nomedal", "hm"}

def get_ioi_results(year: int) -> pd.DataFrame:
    """
    Scrapes IOI results for a given year from the official IOI statistics website.
    
    Args:
        year: The year to scrape results for
        
    Returns:
        DataFrame containing contestant data for the given year
    """
    url = f"{BASE}contestants/{year}"
    print(f"Scraping data for year {year}...")
    
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for year {year}: {e}")
        return pd.DataFrame()
    
    soup = BeautifulSoup(r.text, "html.parser")

    data = []
    current_country = None          # stores the current country (propagates due to rowspan)
    current_country_link = None     # stores the current country link
    span_left = 0                   # number of rows left to propagate the rowspan country

    for row in soup.select("table tr"):
        cols = row.find_all("td")
        if not cols:
            continue

        # 1) Contestant (first column)
        contestant = cols[0].get_text(strip=True)
        if not contestant:
            # update rowspan counter even if the row has no contestant
            if span_left > 0:
                span_left -= 1
            continue

        # 2) Country (only appears in the first row of the block if rowspan > 1)
        a_country = row.select_one('a[href*="countries/"]')
        if a_country:
            td_country = a_country.find_parent("td")
            current_country = a_country.get_text(strip=True)
            current_country_link = urljoin(BASE, a_country["href"])
            try:
                span_left = int(td_country.get("rowspan", "1")) - 1
            except ValueError:
                span_left = 0
        else:
            if span_left > 0:
                span_left -= 1
            # keep the previous country and link if still within rowspan

        country = current_country
        country_link = current_country_link

        # 3) Result: first <td> with gold/silver/bronze/nomedal class
        result_td = row.find(
            lambda tag: tag.name == "td" and set(tag.get("class", [])).intersection(RESULT_CLASSES)
        )
        result = result_td.get_text(strip=True) if result_td else None

        # 4) Codeforces link (if available)
        cf_link = None
        cf_tag = row.find("a", href=True, class_="tableimglink codeforces")
        if cf_tag:
            cf_link = cf_tag["href"]

        if contestant and result:
            data.append({
                "Year": year,
                "Contestant": contestant,
                "Country": country,
                "Country_Link": country_link,
                "Result": result,
                "CF_Link": cf_link
            })

    print(f"Found {len(data)} contestants for year {year}")
    return pd.DataFrame(data)

def get_save_path():
    """
    Determines the save path for the Excel file.
    Uses a common Dropbox structure that works across different users.
    
    Returns:
        Path object pointing to the save directory
    """
    # Try to find Dropbox folder in common locations
    possible_paths = [
        Path.home() / "Dropbox",  # Most common location
        Path("C:/Users") / os.getenv("USERNAME", "") / "Dropbox",  # Windows specific
        Path("/Users") / os.getenv("USER", "") / "Dropbox",  # macOS specific
    ]
    
    # Check if any of these paths exist
    for base_path in possible_paths:
        full_path = base_path / "GTAllocation" / "Data" / "IOI"
        if base_path.exists():
            full_path.mkdir(parents=True, exist_ok=True)
            return full_path
    
    # Fallback: create in current directory
    fallback_path = Path.cwd() / "IOI_Data"
    fallback_path.mkdir(exist_ok=True)
    print(f"Warning: Dropbox folder not found. Using fallback path: {fallback_path}")
    return fallback_path

# Main execution
if __name__ == "__main__":
    print("Starting IOI results collection...")
    
    # Collect results from 2011–2025
    dfs = []
    for year in range(2011, 2026):  # This includes 2025
        df = get_ioi_results(year)
        if not df.empty:
            dfs.append(df)
        else:
            print(f"No data found for year {year}")
    
    if not dfs:
        print("No data collected. Exiting.")
        exit(1)
    
    # Combine all dataframes
    final_df = pd.concat(dfs, ignore_index=True)
    print(f"Total contestants collected: {len(final_df)}")
    print(f"Years covered: {sorted(final_df['Year'].unique())}")
    
    # Get save path
    save_dir = get_save_path()
    out_path = save_dir / "IOI_2011_2025.xlsx"
    
    # Save to Excel
    try:
        final_df.to_excel(out_path, index=False)
        print(f"File successfully saved at: {out_path}")
        
        # Display basic statistics
        print("\nData Summary:")
        print(f"- Total contestants: {len(final_df)}")
        print(f"- Countries: {final_df['Country'].nunique()}")
        print(f"- Years: {final_df['Year'].min()}-{final_df['Year'].max()}")
        print("\nResults distribution:")
        print(final_df['Result'].value_counts())
        
    except Exception as e:
        print(f"Error saving file: {e}")
        # Try alternative path
        alt_path = Path.cwd() / "IOI_2011_2025.xlsx"
        final_df.to_excel(alt_path, index=False)
        print(f"File saved to alternative location: {alt_path}")

Starting IOI results collection...
Scraping data for year 2011...
Found 307 contestants for year 2011
Scraping data for year 2012...
Found 314 contestants for year 2012
Scraping data for year 2013...
Found 303 contestants for year 2013
Scraping data for year 2014...
Found 311 contestants for year 2014
Scraping data for year 2015...
Found 322 contestants for year 2015
Scraping data for year 2016...
Found 312 contestants for year 2016
Scraping data for year 2017...
Found 311 contestants for year 2017
Scraping data for year 2018...
Found 339 contestants for year 2018
Scraping data for year 2019...
Found 331 contestants for year 2019
Scraping data for year 2020...
Found 347 contestants for year 2020
Scraping data for year 2021...
Found 356 contestants for year 2021
Scraping data for year 2022...
Found 357 contestants for year 2022
Scraping data for year 2023...
Found 355 contestants for year 2023
Scraping data for year 2024...
Found 366 contestants for year 2024
Scraping data for year 2025